# FAISS Vector Retrieval Notebook

This notebook is prepared for running retrieval after `index.faiss` and `payloads.jsonl` finish downloading.

Expected index directory layout:

```text
data/faiss_index/
  index.faiss
  payloads.jsonl
  id_map.json        # optional, but recommended if available
```

Run the cells from top to bottom. If downloads are not finished yet, the preflight cell will tell you what is still missing.

## 1. Environment setup

In [ ]:
# Optional: install runtime dependencies if your environment does not have them yet.
# Uncomment and run once if needed.
%pip install -q faiss-cpu sentence-transformers pandas openai


In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import logging

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # Useful if the notebook is launched from notebooks/.
    PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print('HF Hub token detected in environment.')
else:
    logging.getLogger('huggingface_hub.utils._http').setLevel(logging.ERROR)
    print('HF_TOKEN not set; suppressing the Hugging Face unauthenticated-request warning.')

print('Project root:', PROJECT_ROOT)
print('src on path:', SRC_DIR.exists())


Project root: d:\Uni_Project\Text_Mining\Project
src on path: True


## 2. Configure artifact paths and retrieval settings

Change `INDEX_DIR` if you download the FAISS files somewhere else.

In [ ]:
# Directory containing index.faiss + payloads.jsonl (+ optional id_map.json)
INDEX_DIR = PROJECT_ROOT / 'data' / 'faiss_index'

# Must match the embedding model used to build index.faiss.
EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'

TOP_K = 30                 # candidates pulled from FAISS before reranking/dedup
TOP_N = 10                 # final chunks returned
SCORE_THRESHOLD = 0.30
EXPAND_UNITS = False       # keep the notebook light on Colab RAM
FILTER_PROFILE = 'broad'   # current_law | broad | historical

INDEX_DIR


WindowsPath('d:/Uni_Project/Text_Mining/Project/data/faiss_index')

## 3. Preflight: wait until downloads are complete

In [ ]:
phase_t0 = time.perf_counter()
required_files = [INDEX_DIR / 'index.faiss', INDEX_DIR / 'payloads.jsonl']
optional_files = [INDEX_DIR / 'id_map.json']

missing = [p for p in required_files if not p.exists()]
if missing:
    print('Downloads are not ready yet. Missing:')
    for p in missing:
        print(' -', p)
else:
    print('Required files found.')
    for p in required_files + optional_files:
        if p.exists():
            print(f'{p.name}: {p.stat().st_size / 1024 / 1024:.2f} MB')
        else:
            print(f'{p.name}: not found (optional)')

print(f'Preflight completed in {time.perf_counter() - phase_t0:.2f}s')


Required files found.
index.faiss: 5911.63 MB
payloads.jsonl: 4836.18 MB
id_map.json: 60.91 MB


## 4. Load the FAISS store and build retriever

In [ ]:
if missing:
    raise FileNotFoundError('Download index.faiss and payloads.jsonl before running this cell.')

import json
import sqlite3
import time
from pathlib import Path

import numpy as np

from retrieval.config import VectorIndexConfig
from retrieval.embeddings import SentenceTransformerEmbedder
from retrieval.retriever import VectorRetriever
from retrieval.stores import SearchHit, payload_matches


class SQLitePayloadFaissVectorStore:
    """FAISS store with a SQLite payload cache.

    Why this is faster than the previous LazyFaissVectorStore:
    - no sparse-offset scan is repeated on every notebook startup;
    - payloads are fetched in one indexed SQLite query per FAISS result batch;
    - no repeated open/seek/read loops against the 4.8 GB JSONL file.

    The first run creates payload_cache.sqlite from payloads.jsonl once. Later runs reuse
    it as long as payloads.jsonl has not changed.
    """

    def __init__(self, *, index, dimension: int, index_dir: Path, payloads_path: Path, cache_path: Path) -> None:
        self.index = index
        self.dimension = dimension
        self.index_dir = index_dir
        self.payloads_path = payloads_path
        self.cache_path = cache_path
        self._conn = sqlite3.connect(str(cache_path))
        self._conn.execute('PRAGMA query_only = ON')

    @classmethod
    def load(cls, index_dir: Path) -> 'SQLitePayloadFaissVectorStore':
        import faiss

        index_path = index_dir / 'index.faiss'
        payloads_path = index_dir / 'payloads.jsonl'
        cache_path = index_dir / 'payload_cache.sqlite'

        if not index_path.exists():
            raise FileNotFoundError(f'FAISS index not found at {index_path}')
        if not payloads_path.exists():
            raise FileNotFoundError(f'Payload file not found at {payloads_path}')

        cls._ensure_payload_cache(payloads_path, cache_path)

        t0 = time.perf_counter()
        read_flags = getattr(faiss, 'IO_FLAG_MMAP', 0) | getattr(faiss, 'IO_FLAG_READ_ONLY', 0)
        try:
            index = faiss.read_index(str(index_path), read_flags)
        except TypeError:
            index = faiss.read_index(str(index_path))
        print(f'FAISS index loaded in {time.perf_counter() - t0:.2f}s')

        return cls(
            index=index,
            dimension=index.d,
            index_dir=index_dir,
            payloads_path=payloads_path,
            cache_path=cache_path,
        )

    @staticmethod
    def _ensure_payload_cache(payloads_path: Path, cache_path: Path) -> None:
        payload_stat = payloads_path.stat()
        meta = {
            'payload_mtime_ns': str(payload_stat.st_mtime_ns),
            'payload_size': str(payload_stat.st_size),
        }

        if cache_path.exists():
            try:
                conn = sqlite3.connect(str(cache_path))
                rows = dict(conn.execute('SELECT key, value FROM meta').fetchall())
                conn.close()
                if rows == meta:
                    print(f'Reusing SQLite payload cache: {cache_path.name}')
                    return
            except Exception:
                pass
            cache_path.unlink(missing_ok=True)

        tmp_path = cache_path.with_suffix('.sqlite.tmp')
        tmp_path.unlink(missing_ok=True)
        print('Building SQLite payload cache once. This may take a while for the 4.8 GB JSONL, but future starts will skip it.')
        t0 = time.perf_counter()
        conn = sqlite3.connect(str(tmp_path))
        conn.execute('PRAGMA journal_mode = OFF')
        conn.execute('PRAGMA synchronous = OFF')
        conn.execute('PRAGMA temp_store = MEMORY')
        conn.execute('CREATE TABLE payloads (line_no INTEGER PRIMARY KEY, payload TEXT NOT NULL)')
        conn.execute('CREATE TABLE meta (key TEXT PRIMARY KEY, value TEXT NOT NULL)')

        batch = []
        with payloads_path.open('r', encoding='utf-8') as f:
            for line_no, line in enumerate(f):
                line = line.strip()
                if line:
                    batch.append((line_no, line))
                if len(batch) >= 10000:
                    conn.executemany('INSERT INTO payloads(line_no, payload) VALUES (?, ?)', batch)
                    batch.clear()
        if batch:
            conn.executemany('INSERT INTO payloads(line_no, payload) VALUES (?, ?)', batch)

        conn.executemany('INSERT INTO meta(key, value) VALUES (?, ?)', list(meta.items()))
        conn.commit()
        conn.close()
        tmp_path.replace(cache_path)
        print(f'Built SQLite payload cache in {time.perf_counter() - t0:.2f}s')

    def _load_payloads(self, line_nos: list[int]) -> dict[int, dict]:
        if not line_nos:
            return {}
        placeholders = ','.join('?' for _ in line_nos)
        rows = self._conn.execute(
            f'SELECT line_no, payload FROM payloads WHERE line_no IN ({placeholders})',
            line_nos,
        ).fetchall()
        return {int(line_no): json.loads(payload) for line_no, payload in rows}

    def _iter_payloads(self):
        for line_no, payload_text in self._conn.execute('SELECT line_no, payload FROM payloads ORDER BY line_no'):
            yield int(line_no), json.loads(payload_text)

    def search(self, vector: list[float], *, limit: int, score_threshold: float | None = None, filters: dict | None = None) -> list[SearchHit]:
        if self.index.ntotal == 0:
            return []

        t_faiss = time.perf_counter()
        query = np.array([vector], dtype=np.float32)
        # Broad/historical/current_law filters are cheap validity filters, but they still
        # require payload inspection. Keep compensation modest to avoid loading hundreds
        # of payloads for every query.
        search_limit = limit * 3 if filters else limit
        search_limit = min(max(search_limit, limit), self.index.ntotal)
        scores, indices = self.index.search(query, search_limit)
        faiss_time = time.perf_counter() - t_faiss

        valid_pairs = [(float(score), int(idx)) for score, idx in zip(scores[0], indices[0]) if idx >= 0]
        if score_threshold is not None:
            valid_pairs = [(score, idx) for score, idx in valid_pairs if score >= score_threshold]

        t_payload = time.perf_counter()
        payloads = self._load_payloads([idx for _, idx in valid_pairs])
        payload_time = time.perf_counter() - t_payload

        hits: list[SearchHit] = []
        for score, idx in valid_pairs:
            payload = payloads.get(idx, {})
            if filters and not payload_matches(payload, filters):
                continue

            point_id = str(payload.get('chunk_id') or idx)
            hits.append(SearchHit(point_id=point_id, score=score, payload=payload))
            if len(hits) >= limit:
                break

        print(f'FAISS search: {faiss_time:.3f}s | payload batch load: {payload_time:.3f}s | inspected: {len(valid_pairs)}')
        return hits

    def scroll(self, filters: dict, limit: int) -> list[SearchHit]:
        hits: list[SearchHit] = []
        for line_no, payload in self._iter_payloads():
            if payload_matches(payload, filters):
                point_id = str(payload.get('chunk_id') or line_no)
                hits.append(SearchHit(point_id=point_id, score=0.0, payload=payload))
                if len(hits) >= limit:
                    break
        return hits

    @property
    def total_vectors(self) -> int:
        return self.index.ntotal

    def close(self) -> None:
        self._conn.close()


load_t0 = time.perf_counter()
config = VectorIndexConfig(
    embedding_model=EMBEDDING_MODEL,
    top_k=TOP_K,
    top_n=TOP_N,
    score_threshold=SCORE_THRESHOLD,
    expand_units=EXPAND_UNITS,
 )

store = SQLitePayloadFaissVectorStore.load(INDEX_DIR)
embedder = SentenceTransformerEmbedder(
    EMBEDDING_MODEL,
    query_prefix=config.query_prefix,
    passage_prefix=config.passage_prefix,
 )
retriever = VectorRetriever(config=config, embedder=embedder, store=store)

print(f'Vector retriever ready in {time.perf_counter() - load_t0:.2f}s')
print(f'Loaded FAISS vectors: {store.total_vectors:,}')
print(f'Embedding dimension: {embedder.dimension}')


## 4.1 Optional: export payload cache to CSV (for inspection)

In [ ]:
import csv


def export_payloads_to_csv(csv_path: Path | None = None, limit: int | None = 5000) -> Path:
    """Export the SQLite payloads table to CSV for manual inspection.

    Defaults to the first `limit` rows to keep the CSV small and fast to open.
    Pass limit=None to export the full table (this can be as large as payloads.jsonl).
    """
    csv_path = csv_path or (INDEX_DIR / 'payloads_export.csv')
    t0 = time.perf_counter()

    query = 'SELECT line_no, payload FROM payloads ORDER BY line_no'
    params: tuple = ()
    if limit is not None:
        query += ' LIMIT ?'
        params = (limit,)

    rows = []
    for line_no, payload_text in store._conn.execute(query, params):
        payload = json.loads(payload_text)
        rows.append({'line_no': line_no, **payload})

    # Payload schemas can vary slightly between chunks, so build the CSV header
    # from the union of keys seen across the exported rows.
    fieldnames: list[str] = []
    seen: set[str] = set()
    for row in rows:
        for key in row.keys():
            if key not in seen:
                seen.add(key)
                fieldnames.append(key)

    with csv_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

    print(f'Exported {len(rows):,} rows to {csv_path} in {time.perf_counter() - t0:.2f}s')
    return csv_path


export_payloads_to_csv()


## 4.2 Optional: download / export payload_cache.sqlite

In [ ]:
def export_payload_cache_sqlite(dest_dir: Path | None = None) -> Path:
    """Copy payload_cache.sqlite out of INDEX_DIR and offer it for download.

    - In Google Colab: triggers a browser download via `google.colab.files.download`.
    - If Google Drive is mounted at /content/drive: also copies the file there.
    - Otherwise: copies the file to `dest_dir` (defaults to the project root) so it is
      easy to locate for a manual download/copy.
    """
    import shutil

    src_path = store.cache_path
    if not src_path.exists():
        raise FileNotFoundError(f'payload_cache.sqlite not found at {src_path}')

    t0 = time.perf_counter()
    print(f'Source: {src_path} ({src_path.stat().st_size / 1024 / 1024:.2f} MB)')

    try:
        from google.colab import files as colab_files  # type: ignore
        in_colab = True
    except ImportError:
        colab_files = None
        in_colab = False

    drive_root = Path('/content/drive')
    if in_colab and drive_root.exists():
        drive_dest = drive_root / 'MyDrive' / 'faiss_payload_cache' / src_path.name
        drive_dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_path, drive_dest)
        print(f'Copied to Google Drive: {drive_dest} in {time.perf_counter() - t0:.2f}s')
        return drive_dest

    if in_colab and colab_files is not None:
        print('Triggering browser download via Colab...')
        colab_files.download(str(src_path))
        print(f'Download triggered in {time.perf_counter() - t0:.2f}s')
        return src_path

    dest_dir = dest_dir or PROJECT_ROOT
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / src_path.name
    if dest_path.resolve() != src_path.resolve():
        shutil.copy2(src_path, dest_path)
    print(f'Copied to {dest_path} in {time.perf_counter() - t0:.2f}s')
    return dest_path


export_payload_cache_sqlite()


## 5. Retrieval helper

In [ ]:
def search(query: str, top_n: int = TOP_N, filter_profile: str = FILTER_PROFILE, score_threshold: float | None = SCORE_THRESHOLD):
    search_t0 = time.perf_counter()
    result = retriever.retrieve(
        query,
        filter_profile=filter_profile,
        top_n=top_n,
        score_threshold=score_threshold,
    )
    print(f'Retrieval completed in {time.perf_counter() - search_t0:.2f}s')
    rows = []
    for rank, chunk in enumerate(result.chunks, start=1):
        rows.append({
            'rank': rank,
            'chunk_id': chunk.chunk_id,
            'citation': chunk.citation_anchor or chunk.citation_label,
            'title': chunk.title,
            'unit_type': chunk.unit_type,
            'validity_group': chunk.validity_group,
            'vector_score': round(chunk.vector_score, 4),
            'rerank_score': round(chunk.rerank_score, 4),
            'text': chunk.chunk_text[:700],
        })
    return rows, result

def show_results(rows):
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(json.dumps(row, ensure_ascii=False, indent=2))


## 5.1 Benchmark helper

In [ ]:
def benchmark_search(query: str, repeats: int = 3, filter_profile: str = FILTER_PROFILE):
    timings = []
    for i in range(repeats):
        t0 = time.perf_counter()
        rows, result = search(query, top_n=TOP_N, filter_profile=filter_profile)
        elapsed = time.perf_counter() - t0
        timings.append(elapsed)
        print(f'Run {i + 1}/{repeats}: {elapsed:.2f}s, returned={len(result.chunks)}, candidates={result.total_candidates}')
    avg = sum(timings) / len(timings)
    print(f'Average retrieval time over {repeats} runs: {avg:.2f}s')
    return timings


## 6. Run a query

In [ ]:
query_t0 = time.perf_counter()
query = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động là gì?'
rows, result = search(query, top_n=10, filter_profile='broad')
print('Filter profile used:', result.filter_profile_used)
print('Total candidates:', result.total_candidates)
print('Empty filter warning:', result.empty_filter_warning)
show_results(rows)
print(f'Query phase completed in {time.perf_counter() - query_t0:.2f}s')


## 7. Optional: inspect one full chunk

In [ ]:
if result.chunks:
    chunk = result.chunks[0]
    print('chunk_id:', chunk.chunk_id)
    print('citation:', chunk.citation_anchor or chunk.citation_label)
    print('title:', chunk.title)
    print('scores:', {'vector': chunk.vector_score, 'rerank': chunk.rerank_score})
    print('--- text ---')
    print(chunk.chunk_text)
    print('--- metadata keys ---')
    print(sorted(chunk.metadata.keys()))


## 8. Configure the answer generator (OpenAI-compatible API)

Set `BASE_URL`, `API_KEY`, and `MODEL_NAME` via environment variables so credentials never end up hardcoded in this notebook. Any OpenAI-compatible chat completions endpoint works (OpenAI, Azure OpenAI, vLLM, Together, OpenRouter, etc.).

```bash
export LLM_BASE_URL="https://api.your-provider.com/v1"
export LLM_API_KEY="..."
export LLM_MODEL_NAME="gpt-4o-mini"
```

If these are not set, the notebook still runs in retrieval-only mode; generation cells are skipped.

In [ ]:
BASE_URL = os.environ.get('LLM_BASE_URL', '').strip()
API_KEY = os.environ.get('LLM_API_KEY', '').strip()
MODEL_NAME = os.environ.get('LLM_MODEL_NAME', '').strip()

if not (BASE_URL and API_KEY and MODEL_NAME):
    print('Generator not fully configured. Set LLM_BASE_URL, LLM_API_KEY, LLM_MODEL_NAME env vars to enable answer generation.')
else:
    masked_key = API_KEY[:4] + '...' + API_KEY[-4:] if len(API_KEY) > 8 else '***'
    print('Generator configured:')
    print('  BASE_URL:', BASE_URL)
    print('  MODEL_NAME:', MODEL_NAME)
    print('  API_KEY:', masked_key)


## 9. Generator client and answer-generation helper

In [ ]:
from openai import OpenAI

ANSWER_PROMPT = """Bạn là hệ thống RAG pháp lý Việt Nam.
Chỉ trả lời dựa trên CONTEXT được cung cấp. Không suy diễn, không bịa thêm ngoài CONTEXT.
Nếu CONTEXT không đủ thông tin để trả lời, hãy trả lời: "Không có đủ thông tin trong ngữ cảnh được cung cấp."

QUESTION:
{question}

CONTEXT:
{context}

Trả lời bằng tiếng Việt có dấu, kèm căn cứ pháp lý (citation) nếu có trong CONTEXT:"""


class GeneratorClient:
    """Thin wrapper around any OpenAI-compatible chat completions API."""

    def __init__(self, *, base_url: str, api_key: str, model: str) -> None:
        self.model = model
        self.client = OpenAI(base_url=base_url, api_key=api_key)

    def generate(self, prompt: str, *, temperature: float = 0.0) -> str:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature,
        )
        return (response.choices[0].message.content or '').strip()


generator = None
if BASE_URL and API_KEY and MODEL_NAME:
    generator = GeneratorClient(base_url=BASE_URL, api_key=API_KEY, model=MODEL_NAME)
    print('Generator client ready.')
else:
    print('Generator client not created (missing config). Retrieval-only mode.')


def format_context_for_prompt(chunks) -> str:
    blocks = []
    for rank, chunk in enumerate(chunks, start=1):
        citation = chunk.citation_anchor or chunk.citation_label or chunk.chunk_id
        blocks.append(f'[{rank}] {citation} - {chunk.title}\n{chunk.chunk_text}')
    return '\n\n'.join(blocks)


def generate_answer(query: str, chunks) -> str:
    if generator is None:
        raise RuntimeError('Generator is not configured. Set LLM_BASE_URL, LLM_API_KEY, LLM_MODEL_NAME.')
    context = format_context_for_prompt(chunks)
    prompt = ANSWER_PROMPT.format(question=query, context=context)
    gen_t0 = time.perf_counter()
    answer = generator.generate(prompt)
    print(f'Generation completed in {time.perf_counter() - gen_t0:.2f}s')
    return answer


## 10. Full RAG pipeline: retrieve + generate

In [ ]:
def ask(query: str, top_n: int = TOP_N, filter_profile: str = FILTER_PROFILE, score_threshold: float | None = SCORE_THRESHOLD):
    """Run the whole retrieval system end to end: retrieve citation-ready chunks, then generate a grounded answer."""
    rows, result = search(query, top_n=top_n, filter_profile=filter_profile, score_threshold=score_threshold)
    print('Filter profile used:', result.filter_profile_used)
    print('Total candidates:', result.total_candidates)
    print('Empty filter warning:', result.empty_filter_warning)
    show_results(rows)

    if not result.chunks:
        print('No chunks retrieved above the score threshold; skipping generation.')
        return {'query': query, 'answer': None, 'result': result}

    if generator is None:
        print('Generator not configured; returning retrieval-only result.')
        return {'query': query, 'answer': None, 'result': result}

    answer = generate_answer(query, result.chunks)
    print('\n--- Answer ---')
    print(answer)
    print('\n--- Citations used ---')
    for rank, chunk in enumerate(result.chunks, start=1):
        print(f'[{rank}] {chunk.citation_anchor or chunk.citation_label} - {chunk.title}')
    return {'query': query, 'answer': answer, 'result': result}


pipeline_query = 'Điều kiện để người lao động đơn phương chấm dứt hợp đồng lao động là gì?'
pipeline_output = ask(pipeline_query, top_n=10, filter_profile='broad')


## 11. Optional: run the full pipeline over a benchmark sample

In [ ]:
import random


def run_benchmark_sample(qa_path: Path | None = None, sample_size: int = 10, filter_profile: str = FILTER_PROFILE, seed: int = 42):
    """Run retrieval (+ generation, if configured) over a random sample of qa_final.jsonl.

    Reports per-question retrieval hit (whether any ground-truth chunk/provision/document id
    appears among the retrieved results) plus an aggregate hit rate and average latency.
    Unanswerable questions (empty ground_truth) are excluded from the hit-rate denominator.
    """
    qa_path = qa_path or (PROJECT_ROOT / 'data' / 'benchmark' / 'qa_final.jsonl')
    if not qa_path.exists():
        raise FileNotFoundError(f'Benchmark file not found at {qa_path}')

    with qa_path.open('r', encoding='utf-8') as f:
        all_cases = [json.loads(line) for line in f if line.strip()]

    rng = random.Random(seed)
    sample = rng.sample(all_cases, min(sample_size, len(all_cases)))

    records = []
    latencies = []
    hits = 0
    scored = 0

    for qa in sample:
        question = qa.get('question') or ''
        ground_truth = qa.get('ground_truth') or {}
        gt_ids = set(ground_truth.get('chunk_ids') or []) | set(ground_truth.get('provision_ids') or []) | set(ground_truth.get('document_ids') or [])
        is_unanswerable = qa.get('answer_type') == 'unanswerable' or not gt_ids

        t0 = time.perf_counter()
        _, result = search(question, top_n=TOP_N, filter_profile=filter_profile)
        elapsed = time.perf_counter() - t0
        latencies.append(elapsed)

        retrieved_ids = set()
        for chunk in result.chunks:
            retrieved_ids.update({chunk.chunk_id, chunk.parent_unit_id, chunk.id_str})

        hit = bool(gt_ids & retrieved_ids)
        if not is_unanswerable:
            scored += 1
            if hit:
                hits += 1

        answer = None
        if generator is not None and result.chunks:
            try:
                answer = generate_answer(question, result.chunks)
            except Exception as exc:
                answer = f'[generation error: {exc}]'

        records.append({
            'qa_id': qa.get('qa_id'),
            'question': question,
            'answer_type': qa.get('answer_type'),
            'category': qa.get('category'),
            'is_unanswerable': is_unanswerable,
            'retrieval_hit': hit,
            'total_candidates': result.total_candidates,
            'latency_s': round(elapsed, 3),
            'generated_answer': answer,
        })

    hit_rate = hits / scored if scored else float('nan')
    avg_latency = sum(latencies) / len(latencies) if latencies else float('nan')

    print(f'Sampled {len(sample)} questions ({scored} scored, {len(sample) - scored} unanswerable excluded)')
    print(f'Hit rate: {hit_rate:.2%}' if scored else 'Hit rate: n/a (no scored questions)')
    print(f'Average retrieval latency: {avg_latency:.3f}s')

    return {
        'records': records,
        'hit_rate': hit_rate,
        'avg_latency_s': avg_latency,
        'sample_size': len(sample),
        'scored': scored,
    }


# Uncomment to run (generation requires BASE_URL/API_KEY/MODEL_NAME to be configured):
# benchmark_summary = run_benchmark_sample(sample_size=10)
